# 08 — Paper figures and tables

The numerical pipeline is complete at this stage, and the remaining task is to consolidate the validated outputs into figures and tables that can be reviewed before they are transferred to the manuscript. This notebook does not rerun the optimization experiments. It reads only the machine-readable outputs produced by the validated scaling, ideal-QAOA, finite-sampling and circuit-resource workflows, and derives the paper-facing representations from those data.

This separation is intentional. The figures should remain a deterministic post-processing layer, so changing labels, axes or table layout cannot alter the numerical experiments from which the values originate.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from oec_qaoa.plotting import (
    benchmark_table,
    build_all_paper_outputs,
)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS = ROOT / "results"
OUTPUT = RESULTS / "figures"

required = [
    RESULTS / "scaling" / "resource_scaling.csv",
    RESULTS / "scaling" / "resource_reduction_ratios.csv",
    RESULTS / "qaoa" / "ideal_qaoa_summary.csv",
    RESULTS / "qaoa" / "finite_sampling_summary.csv",
    RESULTS / "hardware" / "connectivity_pressure.csv",
]

missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Validated workflow outputs are missing:\n"
        + "\n".join(str(path) for path in missing)
    )

## Frozen benchmark summary

The benchmark table collects the quantities needed to identify the three non-trivial QAOA cases without repeating the details of their construction in every result figure. In particular, it separates semantic path-selection variables from slack variables and reports the quadratic-interaction structure used in the cost Hamiltonian.

In [ ]:
benchmarks = benchmark_table()
benchmarks

## Validated numerical inputs

The tables below are loaded directly from the workflow outputs. Their purpose in this notebook is inspection rather than recomputation: the resource-scaling table summarizes the formulation-level growth, the ideal-QAOA table records the five-restart statevector study, the finite-sampling table quantifies measurement uncertainty, and the connectivity table provides the noise-free hardware-facing metrics.

In [ ]:
scaling = pd.read_csv(
    RESULTS / "scaling" / "resource_scaling.csv"
)
ratios = pd.read_csv(
    RESULTS / "scaling" / "resource_reduction_ratios.csv"
)
ideal = pd.read_csv(
    RESULTS / "qaoa" / "ideal_qaoa_summary.csv"
)
finite = pd.read_csv(
    RESULTS / "qaoa" / "finite_sampling_summary.csv"
)
connectivity = pd.read_csv(
    RESULTS / "hardware" / "connectivity_pressure.csv"
)

{
    "scaling_rows": len(scaling),
    "ratio_rows": len(ratios),
    "ideal_rows": len(ideal),
    "finite_sampling_rows": len(finite),
    "connectivity_rows": len(connectivity),
}

## Figure set

The figure set is organized around the principal questions of the numerical study rather than around the implementation order. The first group quantifies how the path reduction changes the size of the binary model; the second reports ideal-QAOA behavior; the third isolates finite-shot effects; and the final figure shows how the remaining QUBO interactions align with representative connectivity graphs.

The generated images are review artifacts. Their numerical contents are fixed by the validated CSV files, while typography, panel organization and captions can still be adjusted before insertion into the paper.

In [ ]:
build_all_paper_outputs(
    results_root=RESULTS,
    output_dir=OUTPUT,
)

sorted(path.name for path in OUTPUT.iterdir())

## Resource reduction

The resource-scaling figures should be read together. The absolute logical-variable and coupling counts show how quickly the direct edge encoding grows with the number of services, while the path-to-edge ratios make the reduction induced by candidate-path preprocessing explicit. A ratio below one is a formulation-level reduction only; it does not include the classical cost of constructing the candidate sets and should not be interpreted as a quantum speedup.

In [ ]:
pd.read_csv(
    OUTPUT / "table_benchmarks.csv"
)

## Ideal QAOA and finite sampling

The ideal-QAOA figures retain both optimal-state and feasible-state probabilities because they answer different operational questions. A low optimal-state probability indicates that exact global optima are difficult to sample, whereas a substantially larger feasible-state probability means that the variational state can still assign non-negligible mass to physically admissible OEC allocations.

The finite-sampling figures then show how these ideal probabilities translate into practical measurement budgets. In particular, the standard deviation of the sampled energy can become small before the exact optimum is observed reliably. This is the quantitative basis for keeping expectation estimation and optimum detection separate in the manuscript discussion.

In [ ]:
ideal[
    [
        "benchmark_id",
        "depth",
        "expected_energy_mean",
        "optimal_probability_mean",
        "feasible_probability_mean",
        "n_evaluations_mean",
    ]
]

## Hardware-resource interpretation

The connectivity figure does not model noise and does not reproduce a particular processor. It compares the same logical interaction graph with generic all-to-all, grid, ring and line connectivity. The resulting mean shortest-path distances complement the logical qubit and (ZZ)-term counts by showing that substantial variable reduction can coexist with a dense residual interaction graph.

This distinction should remain explicit in the paper: path reduction lowers the size of the binary model, but the shared-resource penalties can still generate interactions that are expensive to realize on sparse-connectivity hardware.

In [ ]:
connectivity[
    [
        "benchmark_id",
        "topology",
        "directly_supported_fraction",
        "mean_interaction_distance",
        "max_interaction_distance",
        "distance_excess_sum",
    ]
]

## Manuscript transfer gate

No numerical statement should be copied into the paper from an intermediate console output or from memory. The manuscript update should use the tables produced in this notebook, together with the underlying validated CSV files, as the numerical source of record.

Before transferring the results, the final review should check four points: the plotted values against the corresponding tables, the caption language against the exact experiment definition, the distinction between formulation reduction and QAOA performance, and the distinction between ideal-state probabilities, finite-shot observation rates and hardware-connectivity proxies. Once these checks are complete, the numerical sections of the manuscript can be unfrozen.